In [1]:
import fitz
import numpy as np
import faiss

from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer

In [2]:
def extract_text_from_pdf(pdf_path):
    document = fitz.open(pdf_path)

    all_text = ""

    for page in document:
        all_text += page.get_text()

    return all_text

In [3]:
pdf_path = "../data/Policies/KYC_Policy.pdf"

all_text = extract_text_from_pdf(pdf_path)

In [4]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = text_splitter.split_text(all_text)

In [5]:
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [6]:
embeddings = model.encode(chunks)

In [7]:
embedding_matrix = np.array(embeddings).astype("float32")

In [8]:
print(embedding_matrix.shape)

(16, 384)


In [9]:
dimension = embedding_matrix.shape[1]

index = faiss.IndexFlatL2(dimension)

In [10]:
index.add(embedding_matrix)

In [11]:
print("Number of vectors stored:", index.ntotal)

Number of vectors stored: 16


In [12]:
query = "What documents are required for customer verification?"

In [13]:
query_embedding = model.encode([query])
query_embedding = np.array(query_embedding).astype("float32")

In [14]:
distances, indices = index.search(query_embedding, k=3)

In [15]:
print(indices)

[[7 5 8]]


In [16]:
for idx in indices[0]:
    print("=" * 80)
    print(f"Chunk {idx + 1}")
    print("=" * 80)
    print(chunks[idx])
    print()

Chunk 8
include: 
• Obtaining senior management approval prior to establishing or continuing the relationship 
• Establishing the source of wealth and source of funds through independent verification 
• Conducting more frequent and intensive ongoing monitoring of the relationship 
• Undertaking enhanced Beneficial Ownership verification, including for layered corporate structures 
• Increasing the frequency of periodic KYC review to not less than once every two years 
7. Required Customer Documents 
The following table summarizes the standard documentation required for Customer Verification across principal 
customer categories. 
ABC National Bank – Know Your Customer (KYC) Policy 
Internal Use Only 
Version 1.0  |  Effective January 2025  |  Page 4 of 6 
Customer Category 
Proof of Identity 
Proof of Address 
Additional Documents 
Individual (Resident) 
Aadhaar, Passport, Voter 
ID, or Driving Licence 
Aadhaar or utility bill not 
older than 2 months 
PAN or Form 60; recent 
photograp